In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [17]:
load_dotenv()

True

In [18]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [19]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [20]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [21]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [22]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [23]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football coach bring a ladder to the library?\n\nBecause he heard the scores were high!',
 'explanation': 'This joke plays on the double meanings of the words "scores" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Scores" (Football Context):** To a football coach, "scores" refers to the points accumulated in a game. For example, "The final score was 21-14."\n2.  **"High Scores" (Football Context):** If a coach hears "the scores were high," he would likely interpret this as meaning there were a lot of points scored, perhaps indicating an exciting game or a very good performance by a team.\n3.  **"Scores" (Library/Music Context):** In a library, "scores" can also refer to **musical scores** – the written notation of a piece of music (like sheet music for an orchestra or a choir). Libraries often have large collections of these.\n4.  **"High" (Physical Context):** When something is "high" in a physical sense, it means it\'s located at a great 

In [24]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach bring a ladder to the library?\n\nBecause he heard the scores were high!', 'explanation': 'This joke plays on the double meanings of the words "scores" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Scores" (Football Context):** To a football coach, "scores" refers to the points accumulated in a game. For example, "The final score was 21-14."\n2.  **"High Scores" (Football Context):** If a coach hears "the scores were high," he would likely interpret this as meaning there were a lot of points scored, perhaps indicating an exciting game or a very good performance by a team.\n3.  **"Scores" (Library/Music Context):** In a library, "scores" can also refer to **musical scores** – the written notation of a piece of music (like sheet music for an orchestra or a choir). Libraries often have large collections of these.\n4.  **"High" (Physical Context):** When something is "high" in a physical sense, it means it\'s 

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach bring a ladder to the library?\n\nBecause he heard the scores were high!', 'explanation': 'This joke plays on the double meanings of the words "scores" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Scores" (Football Context):** To a football coach, "scores" refers to the points accumulated in a game. For example, "The final score was 21-14."\n2.  **"High Scores" (Football Context):** If a coach hears "the scores were high," he would likely interpret this as meaning there were a lot of points scored, perhaps indicating an exciting game or a very good performance by a team.\n3.  **"Scores" (Library/Music Context):** In a library, "scores" can also refer to **musical scores** – the written notation of a piece of music (like sheet music for an orchestra or a choir). Libraries often have large collections of these.\n4.  **"High" (Physical Context):** When something is "high" in a physical sense, it means it\'s

In [26]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket (the insect) get kicked out of the cricket match?\n\nBecause he kept chirping, and the umpire thought he was appealing for *every single ball*!',
 'explanation': 'This joke works on a few levels, primarily through a **play on words** and a **misunderstanding**:\n\n1.  **Double Meaning of "Cricket":**\n    *   The joke starts by using "cricket" to refer to the **insect** (the chirping bug).\n    *   It then places this insect in a "cricket match," referring to the **sport** of cricket. This immediate double meaning sets up the premise.\n\n2.  **What a Cricket (Insect) Does:**\n    *   A real cricket is known for its distinctive, continuous **"chirping"** sound, which it makes by rubbing its wings together.\n\n3.  **What "Appealing" Means in Cricket (Sport):**\n    *   In the sport of cricket, when the fielding team (the bowlers and fielders) believes a batsman should be out, they "appeal" to the umpire. This is a formal request for the 

In [27]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket (the insect) get kicked out of the cricket match?\n\nBecause he kept chirping, and the umpire thought he was appealing for *every single ball*!', 'explanation': 'This joke works on a few levels, primarily through a **play on words** and a **misunderstanding**:\n\n1.  **Double Meaning of "Cricket":**\n    *   The joke starts by using "cricket" to refer to the **insect** (the chirping bug).\n    *   It then places this insect in a "cricket match," referring to the **sport** of cricket. This immediate double meaning sets up the premise.\n\n2.  **What a Cricket (Insect) Does:**\n    *   A real cricket is known for its distinctive, continuous **"chirping"** sound, which it makes by rubbing its wings together.\n\n3.  **What "Appealing" Means in Cricket (Sport):**\n    *   In the sport of cricket, when the fielding team (the bowlers and fielders) believes a batsman should be out, they "appeal" to the umpire. This is a form

In [28]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket (the insect) get kicked out of the cricket match?\n\nBecause he kept chirping, and the umpire thought he was appealing for *every single ball*!', 'explanation': 'This joke works on a few levels, primarily through a **play on words** and a **misunderstanding**:\n\n1.  **Double Meaning of "Cricket":**\n    *   The joke starts by using "cricket" to refer to the **insect** (the chirping bug).\n    *   It then places this insect in a "cricket match," referring to the **sport** of cricket. This immediate double meaning sets up the premise.\n\n2.  **What a Cricket (Insect) Does:**\n    *   A real cricket is known for its distinctive, continuous **"chirping"** sound, which it makes by rubbing its wings together.\n\n3.  **What "Appealing" Means in Cricket (Sport):**\n    *   In the sport of cricket, when the fielding team (the bowlers and fielders) believes a batsman should be out, they "appeal" to the umpire. This is a for

### Time Travel

In [1]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

NameError: name 'workflow' is not defined

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f15d96a-02e6-69e6-8001-5bb2d5c6e2c1"}})

EmptyInputError: Received no input for __start__

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stumps with two bails on top, which the bowler tries to hit and the batsman defends. You absolutely cannot play a game of cricket without a wicket (or two, one at each end of the pitch).\n\n2.  **Meaning 2 (General English Term):** A "wicket" can also mean a small gate or entrance, a turnstile, or a window where tickets are sold (like a ticket wicket).\n\n**How the joke works:**\n\n*   The setup asks why a cricket team got locked out of the stadium. This immediately makes you think about *entering* the stadium.\n*   The punchline, "Because they couldn\'t find the **wicket**!", cleverly uses the second meaning. If th

#### Updating State

In [ ]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-06-01T08:53:08.048692+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, tasks=(PregelTask(id='872c0b90-014c-8496-9675-b150967bd288', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stum

### Fault Tolerance

In [38]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [39]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [40]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [41]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [42]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [43]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [44]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f17a215-cb88-69a2-8003-3ad0568b2143'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-07T16:31:53.330320+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f17a215-cb88-69a1-8002-255adae53fda'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f17a215-cb88-69a1-8002-255adae53fda'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-07T16:31:53.330320+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f17a212-e9e3-6212-8001-bc6c25172f00'}}, tasks=(PregelTask(id='6e4b9eac-8e56-3256-9a8e-c822d48c0fa